# dl_helper Kaggle 剩余发布门禁

按顺序运行全部单元：sweep 真机门禁、sklearn incremental CPU smoke、脱敏检查和证据打包。

In [ ]:
import os
from datetime import datetime, timezone

os.environ['DL_HELPER_GIT_REPO'] = 'https://github.com/lhiqwj173/dl_helper.git'
os.environ['DL_HELPER_GIT_REF'] = '65f1063ffe3ebef746d080c503bfc27b29999829'
os.environ['DL_HELPER_MNIST_PATH'] = '/kaggle/input/datasets/vikramtiwari/mnist-numpy/mnist.npz'
os.environ['ALIST_HOST'] = 'https://tmsatws.kdns.fr'
os.environ['ALIST_BASE_PATH'] = '/dl-helper/release-gate'
os.environ['WECOM_TO_USER'] = '@all'
run_stamp = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
os.environ['DL_HELPER_RUN_STAMP'] = run_stamp
os.environ['DL_HELPER_SWEEP_ID'] = f'kaggle-toy-sweep-{run_stamp}'
os.environ['DL_HELPER_SKLEARN_RUN_ID'] = f'sklearn-incremental-smoke-{run_stamp}'
print('代码版本和本次唯一门禁 ID 已设置:', run_stamp)

In [ ]:
import os
import subprocess
import sys

repo_dir = '/kaggle/working/dl-helper'
if os.path.exists(repo_dir):
    raise RuntimeError(f'目录已存在，请使用全新 Kaggle Session: {repo_dir}')

def checked(argv, *, cwd=None, expected=(0,)):
    proc = subprocess.run(argv, cwd=cwd, capture_output=True, text=True, encoding='utf-8')
    if proc.stdout:
        print(proc.stdout, end='')
    if proc.stderr:
        print(proc.stderr, file=sys.stderr, end='')
    if proc.returncode not in expected:
        raise RuntimeError(f'命令退出码 {proc.returncode}，预期 {expected}: {argv}')
    return proc

checked(['git', 'clone', os.environ['DL_HELPER_GIT_REPO'], repo_dir])
checked(['git', 'checkout', os.environ['DL_HELPER_GIT_REF']], cwd=repo_dir)
head = checked(['git', 'rev-parse', 'HEAD'], cwd=repo_dir).stdout.strip()
if head.lower() != os.environ['DL_HELPER_GIT_REF'].lower():
    raise RuntimeError(f'checkout HEAD 不匹配: {head}')
os.environ['DL_HELPER_REPO_DIR'] = repo_dir
checked([sys.executable, f'{repo_dir}/envs/kaggle_bootstrap.py'])
print('[bootstrap] OK:', head)

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, '/kaggle/working/dl-helper')
import yaml
from dl_helper.training.config import default_schema

config_root = Path('/kaggle/working/release-gate-config')
project_dir = '/kaggle/working/dl-helper/examples'
sweep_root = config_root / 'sweep'
variant_root = sweep_root / 'variants'
variant_root.mkdir(parents=True, exist_ok=False)

remote = {'type': 'alist', 'host': os.environ['ALIST_HOST'], 'base_path': os.environ['ALIST_BASE_PATH'], 'user_secret_key': 'ALIST_USER', 'password_secret_key': 'ALIST_PWD', 'connect_timeout_seconds': 10, 'read_timeout_seconds': 60, 'max_attempts': 3, 'async_upload': False, 'failure_policy': 'required'}
notifications = {'type': 'wecom', 'corp_id_secret_key': 'WECOM_CORP_ID', 'corp_secret_key': 'WECOM_CORP_SECRET', 'agent_id_secret_key': 'WECOM_AGENT_ID', 'to_user': os.environ['WECOM_TO_USER'], 'connect_timeout_seconds': 10, 'read_timeout_seconds': 30, 'max_attempts': 3, 'failure_policy': 'required'}

sweep_base = default_schema()
sweep_base['run'].update({'name': 'kaggle-toy-sweep', 'id': None, 'output_root': None, 'source_revision': os.environ['DL_HELPER_GIT_REF']})
sweep_base['experiment'] = {'lr': 0.01}
sweep_base['training'] = {'max_epochs': 2, 'log_every_steps': 10}
sweep_base['backend']['torch'].update({'mixed_precision': 'no', 'deterministic': 'off'})
sweep_base['distributed'] = {'num_processes': 1}
sweep_base['selection'] = {'metric': 'val/loss', 'mode': 'min', 'patience': 20, 'min_delta': 0.0}
sweep_base['checkpoint'] = {'every_epochs': 1, 'every_optimizer_steps': None, 'keep_last': 2}
sweep_base['report']['prediction_splits'] = ['val']
sweep_base['remote'] = remote
sweep_base['notifications'] = notifications

with (sweep_root / 'base.yaml').open('w', encoding='utf-8') as f:
    yaml.safe_dump(sweep_base, f, allow_unicode=True, sort_keys=False)
for name, lr in [('lr-1e-2', 0.01), ('lr-1e-3', 0.001)]:
    with (variant_root / f'{name}.yaml').open('w', encoding='utf-8') as f:
        yaml.safe_dump({'experiment': {'lr': lr}}, f, allow_unicode=True, sort_keys=False)
sweep_manifest = {'schema_version': 1, 'sweep': {'id': os.environ['DL_HELPER_SWEEP_ID'], 'experiment': 'experiments.toy_multiclass_resumable:build_experiment', 'base_config': './base.yaml', 'comparison_metric': 'val/loss', 'mode': 'min', 'trials': [{'name': 'lr-1e-2', 'variant': './variants/lr-1e-2.yaml'}, {'name': 'lr-1e-3', 'variant': './variants/lr-1e-3.yaml'}]}}
with (sweep_root / 'sweep.yaml').open('w', encoding='utf-8') as f:
    yaml.safe_dump(sweep_manifest, f, allow_unicode=True, sort_keys=False)

sklearn_config = default_schema()
sklearn_config['run'].update({'name': 'sklearn-incremental-smoke', 'id': os.environ['DL_HELPER_SKLEARN_RUN_ID'], 'output_root': None, 'source_revision': os.environ['DL_HELPER_GIT_REF']})
sklearn_config['experiment'] = {'loss': 'log_loss', 'n_classes': 3}
sklearn_config['training'] = {'max_epochs': 2, 'log_every_steps': 1}
sklearn_config['backend'] = {'type': 'sklearn', 'torch': None, 'sklearn': {'fit_mode': 'incremental', 'evaluation_batch_size': 4096, 'n_jobs': None, 'random_state': 'run_seed', 'sample_weight_parameter': None}}
sklearn_config['distributed'] = {'num_processes': 1}
sklearn_config['selection'] = {'metric': 'val/accuracy', 'mode': 'max', 'patience': 20, 'min_delta': 0.0}
sklearn_config['checkpoint'] = {'every_epochs': 1, 'every_optimizer_steps': None, 'keep_last': 3}
sklearn_config['report']['prediction_splits'] = ['val']
sklearn_config['remote'] = remote
sklearn_config['notifications'] = notifications
sklearn_path = config_root / 'sklearn-incremental.yaml'
with sklearn_path.open('w', encoding='utf-8') as f:
    yaml.safe_dump(sklearn_config, f, allow_unicode=True, sort_keys=False)
print('配置已生成:', sweep_root / 'sweep.yaml', sklearn_path)

In [ ]:
import requests
response = requests.get(f"{os.environ['ALIST_HOST']}/api/public/settings", timeout=20)
print('AList HTTPS status:', response.status_code)
if response.status_code != 200:
    raise RuntimeError(f'AList HTTPS 检查失败: {response.status_code}')

## Sweep 真机门禁

In [ ]:
sweep_base_path = str(sweep_root / 'base.yaml')
checked([sys.executable, '-m', 'dl_helper.training.cli', 'train', '--config', sweep_base_path, '--project-dir', project_dir, '--experiment', 'experiments.toy_multiclass_resumable:build_experiment', '--preflight-only'], cwd=repo_dir)
print('sweep preflight exit code: 0')

In [ ]:
sweep_proc = checked([sys.executable, '-m', 'dl_helper.training.cli', 'sweep', '--sweep', str(sweep_root / 'sweep.yaml'), '--project-dir', project_dir], cwd=repo_dir)
print('sweep exit code:', sweep_proc.returncode)

In [ ]:
import hashlib
import json

output_root = Path('/kaggle/working/dl-helper-runs')
sweep_dir = output_root / 'sweeps' / os.environ['DL_HELPER_SWEEP_ID']
sweep_terminal = [p for p in ['sweep-manifest.json', 'pause-manifest.json', 'failure.json'] if (sweep_dir / p).is_file()]
if sweep_terminal != ['sweep-manifest.json']:
    raise RuntimeError(f'sweep 终态不唯一或非成功: {sweep_terminal}')
manifest = json.loads((sweep_dir / 'sweep-manifest.json').read_text(encoding='utf-8'))
if len(manifest.get('ranking', [])) != 2 or not manifest.get('best_trial'):
    raise RuntimeError('sweep ranking/best_trial 不完整')
for rel in ['sweep-manifest.json', 'best-trial.json', 'services/service-manifest.json', 'services/service-audit.jsonl', 'sweep-report/index.html']:
    path = sweep_dir / rel
    if not path.is_file():
        raise FileNotFoundError(path)
    print(rel, hashlib.sha256(path.read_bytes()).hexdigest())
print('best trial:', manifest['best_trial'])

## sklearn incremental CPU smoke

In [ ]:
checked([sys.executable, '-m', 'dl_helper.training.cli', 'train', '--config', str(sklearn_path), '--project-dir', project_dir, '--experiment', 'experiments.sklearn_incremental:build_experiment', '--preflight-only'], cwd=repo_dir)
print('sklearn preflight exit code: 0')

In [ ]:
sklearn_proc = checked([sys.executable, '-m', 'dl_helper.training.cli', 'train', '--config', str(sklearn_path), '--project-dir', project_dir, '--experiment', 'experiments.sklearn_incremental:build_experiment', '--run-id', os.environ['DL_HELPER_SKLEARN_RUN_ID']], cwd=repo_dir)
print('sklearn train exit code:', sklearn_proc.returncode)

In [ ]:
sklearn_dir = output_root / 'runs' / os.environ['DL_HELPER_SKLEARN_RUN_ID']
sklearn_terminal = [p for p in ['run-manifest.json', 'pause-manifest.json', 'failure.json'] if (sklearn_dir / p).is_file()]
if sklearn_terminal != ['run-manifest.json']:
    raise RuntimeError(f'sklearn 终态不唯一或非成功: {sklearn_terminal}')
for rel in ['run-manifest.json', 'evaluation-contract.json', 'services/service-manifest.json', 'services/service-audit.jsonl', 'report/index.html', 'models/last/model.joblib']:
    path = sklearn_dir / rel
    if not path.is_file():
        raise FileNotFoundError(path)
    print(rel, hashlib.sha256(path.read_bytes()).hexdigest())

## 脱敏扫描与证据打包

In [ ]:
import zipfile
from kaggle_secrets import UserSecretsClient

secret_client = UserSecretsClient()
secret_values = [secret_client.get_secret(name).encode('utf-8') for name in ['ALIST_USER', 'ALIST_PWD', 'WECOM_CORP_ID', 'WECOM_CORP_SECRET', 'WECOM_AGENT_ID']]
evidence_roots = [sweep_dir, sklearn_dir]
for root in evidence_roots:
    for path in root.rglob('*'):
        if path.is_file():
            content = path.read_bytes()
            if any(value and value in content for value in secret_values):
                raise RuntimeError(f'证据文件包含 Secret: {path}')
print('Secret scan: PASS')

def write_evidence_zip(zip_path, roots):
    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
        for root, prefix in roots:
            for path in sorted(root.rglob('*')):
                if path.is_file():
                    archive.write(path, arcname=str(Path(prefix) / path.relative_to(root)))
    return hashlib.sha256(Path(zip_path).read_bytes()).hexdigest()

trial_roots = [(output_root / 'runs' / f"{os.environ['DL_HELPER_SWEEP_ID']}--{name}", f'runs/{os.environ["DL_HELPER_SWEEP_ID"]}--{name}') for name in ['lr-1e-2', 'lr-1e-3']]
evidence_stamp = os.environ['DL_HELPER_RUN_STAMP']
sweep_zip = f'/kaggle/working/kaggle-sweep-evidence-{evidence_stamp}.zip'
sklearn_zip = f'/kaggle/working/sklearn-incremental-evidence-{evidence_stamp}.zip'
sweep_sha = write_evidence_zip(sweep_zip, [(sweep_dir, f'sweeps/{os.environ["DL_HELPER_SWEEP_ID"]}'), *trial_roots])
sklearn_sha = write_evidence_zip(sklearn_zip, [(sklearn_dir, f'runs/{os.environ["DL_HELPER_SKLEARN_RUN_ID"]}')])
print('sweep evidence:', sweep_zip, sweep_sha)
print('sklearn evidence:', sklearn_zip, sklearn_sha)